In [3]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 01_feature_engineering.ipynb
# =============================================================================

In [4]:
!pip install duckdb
!pip install polars
!pip install pyarrow
!pip install fastparquet
!pip install tqdm

In [5]:
from pathlib import Path

BASE_PATH = Path(
    r"C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2"
)

print(BASE_PATH)
print(BASE_PATH.exists())

C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2
True


In [6]:
import duckdb
import polars as pl
import pandas as pd

from tqdm import tqdm

In [7]:
for file in sorted(BASE_PATH.rglob("*")):
    print(file)

C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\device.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\email.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\file.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\http.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\LDAP
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\LDAP\2009-12.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\LDAP\2010-01.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\LDAP\2010-02.csv
C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\raw\r4.2\LDAP\2010-03.csv
C:\Users\kabra\OneDrive\

In [8]:
device = pl.read_csv(
    BASE_PATH / "device.csv",
    n_rows=5
)

device

id,date,user,pc,activity
str,str,str,str,str
"""{J1S3-L9UU75BQ-7790ATPL}""","""01/02/2010 07:21:06""","""MOH0273""","""PC-6699""","""Connect"""
"""{N7B5-Y7BB27SI-2946PUJK}""","""01/02/2010 07:37:41""","""MOH0273""","""PC-6699""","""Disconnect"""
"""{U1V9-Z7XT67KV-5649MYHI}""","""01/02/2010 07:59:11""","""HPH0075""","""PC-2417""","""Connect"""
"""{H0Z7-E6GB57XZ-1603MOXD}""","""01/02/2010 07:59:49""","""IIW0249""","""PC-0843""","""Connect"""
"""{L7P2-G4PX02RX-7999GYOY}""","""01/02/2010 08:04:26""","""IIW0249""","""PC-0843""","""Disconnect"""


In [9]:
datasets = {

    "device": BASE_PATH/"device.csv",

    "email": BASE_PATH/"email.csv",

    "file": BASE_PATH/"file.csv",

    "http": BASE_PATH/"http.csv",

    "logon": BASE_PATH/"logon.csv",

    "psychometric": BASE_PATH/"psychometric.csv"

}
summary = []

for name, path in datasets.items():

    df = pl.read_csv(path, n_rows=5)

    summary.append({

        "Dataset": name,

        "Columns": df.columns,

        "Count": len(df.columns)

    })

summary

[{'Dataset': 'device',
  'Columns': ['id', 'date', 'user', 'pc', 'activity'],
  'Count': 5},
 {'Dataset': 'email',
  'Columns': ['id',
   'date',
   'user',
   'pc',
   'to',
   'cc',
   'bcc',
   'from',
   'size',
   'attachments',
   'content'],
  'Count': 11},
 {'Dataset': 'file',
  'Columns': ['id', 'date', 'user', 'pc', 'filename', 'content'],
  'Count': 6},
 {'Dataset': 'http',
  'Columns': ['id', 'date', 'user', 'pc', 'url', 'content'],
  'Count': 6},
 {'Dataset': 'logon',
  'Columns': ['id', 'date', 'user', 'pc', 'activity'],
  'Count': 5},
 {'Dataset': 'psychometric',
  'Columns': ['employee_name', 'user_id', 'O', 'C', 'E', 'A', 'N'],
  'Count': 7}]

In [10]:
OUTPUT_DIR = BASE_PATH.parent.parent / "profiling"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

print(f"Saved to: {OUTPUT_DIR / 'dataset_summary.csv'}")

Saved to: C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\profiling\dataset_summary.csv


In [11]:
# =============================================================================
# CREATE DATASET REGISTRY
# =============================================================================

In [12]:
dataset_registry = {
    "device": BASE_PATH / "device.csv",
    "email": BASE_PATH / "email.csv",
    "file": BASE_PATH / "file.csv",
    "http": BASE_PATH / "http.csv",
    "logon": BASE_PATH / "logon.csv",
    "psychometric": BASE_PATH / "psychometric.csv",
}

In [13]:
for name, path in dataset_registry.items():
    print(f"{name:15} -> {path.exists()}")

device          -> True
email           -> True
file            -> True
http            -> True
logon           -> True
psychometric    -> True


In [14]:
import duckdb

con = duckdb.connect()

dataset_statistics = []

for name, path in dataset_registry.items():

    print(f"Processing {name}...")

    query = f"""
        SELECT COUNT(*) AS rows
        FROM read_csv_auto('{path.as_posix()}')
    """

    rows = con.execute(query).fetchone()[0]

    dataset_statistics.append({
        "Dataset": name,
        "Rows": rows
    })

dataset_statistics

Processing device...
Processing email...
Processing file...
Processing http...
Processing logon...
Processing psychometric...


[{'Dataset': 'device', 'Rows': 405380},
 {'Dataset': 'email', 'Rows': 2629979},
 {'Dataset': 'file', 'Rows': 445581},
 {'Dataset': 'http', 'Rows': 28434423},
 {'Dataset': 'logon', 'Rows': 854859},
 {'Dataset': 'psychometric', 'Rows': 1000}]

In [15]:
schema_information = {}

for name, path in dataset_registry.items():

    print("=" * 80)
    print(f"{name.upper()} DATASET")
    print("=" * 80)

    df = duckdb.sql(
        f"""
        SELECT *
        FROM read_csv_auto('{path.as_posix()}')
        LIMIT 5
        """
    ).df()

    print(df.dtypes)
    print()
    print(df.head())

    schema_information[name] = {
        "columns": list(df.columns),
        "dtypes": {
            c: str(df[c].dtype)
            for c in df.columns
        }
    }

    print("\n")

DEVICE DATASET
id          object
date        object
user        object
pc          object
activity    object
dtype: object

                         id                 date     user       pc    activity
0  {J1S3-L9UU75BQ-7790ATPL}  01/02/2010 07:21:06  MOH0273  PC-6699     Connect
1  {N7B5-Y7BB27SI-2946PUJK}  01/02/2010 07:37:41  MOH0273  PC-6699  Disconnect
2  {U1V9-Z7XT67KV-5649MYHI}  01/02/2010 07:59:11  HPH0075  PC-2417     Connect
3  {H0Z7-E6GB57XZ-1603MOXD}  01/02/2010 07:59:49  IIW0249  PC-0843     Connect
4  {L7P2-G4PX02RX-7999GYOY}  01/02/2010 08:04:26  IIW0249  PC-0843  Disconnect


EMAIL DATASET
id                     object
date           datetime64[us]
user                   object
pc                     object
to                     object
cc                     object
bcc                    object
from                   object
size                    int64
attachments             int64
content                object
dtype: object

                         id             

In [16]:
schema_rows = []

for dataset, info in schema_information.items():

    for column in info["columns"]:

        schema_rows.append({

            "Dataset": dataset,

            "Column": column,

            "Datatype": info["dtypes"][column]

        })

schema_df = pd.DataFrame(schema_rows)

schema_df.head(20)

,Dataset,Column,Datatype
0,device,id,object
1,device,date,object
2,device,user,object
3,device,pc,object
4,device,activity,object
5,email,id,object
6,email,date,datetime64[us]
7,email,user,object
8,email,pc,object
9,email,to,object


In [18]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path(
    r"C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system"
)

# Output directory
PROFILE_DIR = PROJECT_ROOT / "datasets" / "profiling"

PROFILE_DIR.mkdir(parents=True, exist_ok=True)

# Output file
SCHEMA_FILE = PROFILE_DIR / "dataset_schema.csv"

# Save
schema_df.to_csv(
    SCHEMA_FILE,
    index=False
)

print("=" * 70)
print("Dataset Schema Saved Successfully")
print("=" * 70)
print(f"Location : {SCHEMA_FILE}")
print("=" * 70)

Dataset Schema Saved Successfully
Location : C:\Users\kabra\OneDrive\Desktop\insider-threat-behavioral-intelligence-system\datasets\profiling\dataset_schema.csv


In [19]:
column_frequency = (
    schema_df.groupby("Column")
             .size()
             .reset_index(name="Datasets")
             .sort_values("Datasets", ascending=False)
)

column_frequency

,Column,Datasets
10,date,5
14,id,5
15,pc,5
19,user,5
9,content,3
5,activity,2
4,O,1
2,E,1
1,C,1
0,A,1


In [20]:
EVENT_COLUMNS = [
    "timestamp",
    "user",
    "pc",
    "event_type",
    "event_details",
    "source"
]

print(EVENT_COLUMNS)

['timestamp', 'user', 'pc', 'event_type', 'event_details', 'source']


In [21]:
logon = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    activity AS event_details
FROM read_csv_auto('{(BASE_PATH/'logon.csv').as_posix()}')
LIMIT 10
""").df()

logon["event_type"] = "LOGON"

logon["source"] = "logon"

logon = logon[EVENT_COLUMNS]

logon

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 06:49:00,NGF0157,PC-6056,LOGON,Logon,logon
1,01/02/2010 06:50:00,LRR0148,PC-4275,LOGON,Logon,logon
2,01/02/2010 06:53:04,LRR0148,PC-4124,LOGON,Logon,logon
3,01/02/2010 07:00:00,IRM0931,PC-7188,LOGON,Logon,logon
4,01/02/2010 07:00:00,MOH0273,PC-6699,LOGON,Logon,logon
5,01/02/2010 07:07:00,LAP0338,PC-5758,LOGON,Logon,logon
6,01/02/2010 07:08:00,MHH0180,PC-9822,LOGON,Logon,logon
7,01/02/2010 07:08:00,NOB0181,PC-3446,LOGON,Logon,logon
8,01/02/2010 07:13:00,AHC0142,PC-8893,LOGON,Logon,logon
9,01/02/2010 07:14:00,CTR0341,PC-6184,LOGON,Logon,logon


In [22]:
device = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    activity AS event_details
FROM read_csv_auto('{(BASE_PATH/'device.csv').as_posix()}')
LIMIT 10
""").df()

device["event_type"] = "DEVICE"

device["source"] = "device"

device = device[EVENT_COLUMNS]

device

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 07:21:06,MOH0273,PC-6699,DEVICE,Connect,device
1,01/02/2010 07:37:41,MOH0273,PC-6699,DEVICE,Disconnect,device
2,01/02/2010 07:59:11,HPH0075,PC-2417,DEVICE,Connect,device
3,01/02/2010 07:59:49,IIW0249,PC-0843,DEVICE,Connect,device
4,01/02/2010 08:04:26,IIW0249,PC-0843,DEVICE,Disconnect,device
5,01/02/2010 08:17:35,HPH0075,PC-2417,DEVICE,Disconnect,device
6,01/02/2010 08:24:54,HSB0196,PC-8001,DEVICE,Connect,device
7,01/02/2010 08:25:18,RRC0553,PC-6672,DEVICE,Connect,device
8,01/02/2010 08:25:19,MOH0273,PC-6699,DEVICE,Connect,device
9,01/02/2010 08:29:40,MOH0273,PC-6699,DEVICE,Disconnect,device


In [24]:
email_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'email.csv').as_posix()}')
LIMIT 5
""").df()

print(email_sample.columns.tolist())

email_sample

['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']


,id,date,user,pc,to,cc,bcc,from,size,attachments,content
0,{R3I7-S4TX96FG-8219JWFF},2010-02-01 07:11:45,LAP0338,PC-5758,Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...,Nathaniel.Hunter.Heath@dtaa.com,None,Lynn.Adena.Pratt@dtaa.com,25830,0,middle f2 systems 4 july techniques powerful d...
1,{R0R9-E4GL59IK-2907OSWJ},2010-02-01 07:12:16,MOH0273,PC-6699,Odonnell-Gage@bellsouth.net,None,None,MOH68@optonline.net,29942,0,the breaking called allied reservations former...
2,{G2B2-A8XY58CP-2847ZJZL},2010-02-01 07:13:00,LAP0338,PC-5758,Penelope_Colon@netzero.com,None,None,Lynn_A_Pratt@earthlink.net,28780,0,slowly this uncinus winter beneath addition ex...
3,{A3A9-F4TH89AA-8318GFGK},2010-02-01 07:13:17,LAP0338,PC-5758,Judith_Hayden@comcast.net,None,None,Lynn_A_Pratt@earthlink.net,21907,0,400 other difficult land cirrocumulus powered ...
4,{E8B7-C8FZ88UF-2946RUQQ},2010-02-01 07:13:28,MOH0273,PC-6699,Bond-Raymond@verizon.net;Alea_Ferrell@msn.com;...,None,Odonnell-Gage@bellsouth.net,MOH68@optonline.net,17319,0,this kmh october holliswood number advised unu...


In [25]:
for name, path in dataset_registry.items():

    print("\n" + "=" * 80)
    print(f"{name.upper()} DATASET")
    print("=" * 80)

    sample = duckdb.sql(f"""
        SELECT *
        FROM read_csv_auto('{path.as_posix()}')
        LIMIT 5
    """).df()

    print("Columns:")
    print(sample.columns.tolist())

    print("\nSample Data:")
    display(sample.head())


DEVICE DATASET
Columns:
['id', 'date', 'user', 'pc', 'activity']

Sample Data:


,id,date,user,pc,activity
0,{J1S3-L9UU75BQ-7790ATPL},01/02/2010 07:21:06,MOH0273,PC-6699,Connect
1,{N7B5-Y7BB27SI-2946PUJK},01/02/2010 07:37:41,MOH0273,PC-6699,Disconnect
2,{U1V9-Z7XT67KV-5649MYHI},01/02/2010 07:59:11,HPH0075,PC-2417,Connect
3,{H0Z7-E6GB57XZ-1603MOXD},01/02/2010 07:59:49,IIW0249,PC-0843,Connect
4,{L7P2-G4PX02RX-7999GYOY},01/02/2010 08:04:26,IIW0249,PC-0843,Disconnect



EMAIL DATASET
Columns:
['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']

Sample Data:


,id,date,user,pc,to,cc,bcc,from,size,attachments,content
0,{R3I7-S4TX96FG-8219JWFF},2010-02-01 07:11:45,LAP0338,PC-5758,Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...,Nathaniel.Hunter.Heath@dtaa.com,None,Lynn.Adena.Pratt@dtaa.com,25830,0,middle f2 systems 4 july techniques powerful d...
1,{R0R9-E4GL59IK-2907OSWJ},2010-02-01 07:12:16,MOH0273,PC-6699,Odonnell-Gage@bellsouth.net,None,None,MOH68@optonline.net,29942,0,the breaking called allied reservations former...
2,{G2B2-A8XY58CP-2847ZJZL},2010-02-01 07:13:00,LAP0338,PC-5758,Penelope_Colon@netzero.com,None,None,Lynn_A_Pratt@earthlink.net,28780,0,slowly this uncinus winter beneath addition ex...
3,{A3A9-F4TH89AA-8318GFGK},2010-02-01 07:13:17,LAP0338,PC-5758,Judith_Hayden@comcast.net,None,None,Lynn_A_Pratt@earthlink.net,21907,0,400 other difficult land cirrocumulus powered ...
4,{E8B7-C8FZ88UF-2946RUQQ},2010-02-01 07:13:28,MOH0273,PC-6699,Bond-Raymond@verizon.net;Alea_Ferrell@msn.com;...,None,Odonnell-Gage@bellsouth.net,MOH68@optonline.net,17319,0,this kmh october holliswood number advised unu...



FILE DATASET
Columns:
['id', 'date', 'user', 'pc', 'filename', 'content']

Sample Data:


...

,id,date,user,pc,url,content
0,{V1Y4-S2IR20QU-6154HFXJ},2010-02-01 06:55:16,LRR0148,PC-4275,http://msn.com/The_Human_Centipede_First_Seque...,remain representatives consensus concert altho...
1,{Q5R1-T3EF87UE-2395RWZS},2010-02-01 07:00:13,NGF0157,PC-6056,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...,festival off northwards than congestion partne...
2,{X9O1-O0XW52VO-5806RPHG},2010-02-01 07:03:46,NGF0157,PC-6056,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...,long away reorganized baldwin seth business 18...
3,{G5S8-U5OG04TE-5299CCTU},2010-02-01 07:05:26,IRM0931,PC-7188,http://groupon.com/Leonhard_Euler/leonhard/tne...,among german schwein experimental becomes prev...
4,{L0R4-A9DH29VP-4553AUWM},2010-02-01 07:05:52,IRM0931,PC-7188,http://flickr.com/Inauguration_of_Barack_Obama...,kate criteria j 2008 highest 12 include books ...



LOGON DATASET
Columns:
['id', 'date', 'user', 'pc', 'activity']

Sample Data:


,id,date,user,pc,activity
0,{X1D9-S0ES98JV-5357PWMI},01/02/2010 06:49:00,NGF0157,PC-6056,Logon
1,{G2B3-L6EJ61GT-2222RKSO},01/02/2010 06:50:00,LRR0148,PC-4275,Logon
2,{U6Q3-U0WE70UA-3770UREL},01/02/2010 06:53:04,LRR0148,PC-4124,Logon
3,{I0N5-R7NA26TG-6263KNGM},01/02/2010 07:00:00,IRM0931,PC-7188,Logon
4,{D1S0-N6FH62BT-5398KANK},01/02/2010 07:00:00,MOH0273,PC-6699,Logon



PSYCHOMETRIC DATASET
Columns:
['employee_name', 'user_id', 'O', 'C', 'E', 'A', 'N']

Sample Data:


,employee_name,user_id,O,C,E,A,N
0,Calvin Edan Love,CEL0561,40,39,36,19,40
1,Christine Reagan Deleon,CRD0624,26,22,17,39,32
2,Jade Felicia Caldwell,JFC0557,22,16,23,40,33
3,Aquila Stewart Dejesus,ASD0577,40,48,36,14,37
4,Micah Abdul Rojas,MAR0955,36,44,23,44,25


In [26]:
print(sample.columns.tolist())

['employee_name', 'user_id', 'O', 'C', 'E', 'A', 'N']


In [27]:
device_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'device.csv').as_posix()}')
LIMIT 5
""").df()

print(device_sample.columns.tolist())

['id', 'date', 'user', 'pc', 'activity']


In [28]:
logon_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'logon.csv').as_posix()}')
LIMIT 5
""").df()

print(logon_sample.columns.tolist())

['id', 'date', 'user', 'pc', 'activity']


In [29]:
email_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'email.csv').as_posix()}')
LIMIT 5
""").df()

print(email_sample.columns.tolist())

['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']


In [30]:
file_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'file.csv').as_posix()}')
LIMIT 5
""").df()

print(file_sample.columns.tolist())

['id', 'date', 'user', 'pc', 'filename', 'content']


In [31]:
http_sample = duckdb.sql(f"""
SELECT *
FROM read_csv_auto('{(BASE_PATH/'http.csv').as_posix()}')
LIMIT 5
""").df()

print(http_sample.columns.tolist())

['id', 'date', 'user', 'pc', 'url', 'content']


In [32]:
logon_events = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    'LOGON' AS event_type,
    activity AS event_details,
    'logon' AS source
FROM read_csv_auto('{(BASE_PATH/'logon.csv').as_posix()}')
LIMIT 100
""").df()

logon_events.head()

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 06:49:00,NGF0157,PC-6056,LOGON,Logon,logon
1,01/02/2010 06:50:00,LRR0148,PC-4275,LOGON,Logon,logon
2,01/02/2010 06:53:04,LRR0148,PC-4124,LOGON,Logon,logon
3,01/02/2010 07:00:00,IRM0931,PC-7188,LOGON,Logon,logon
4,01/02/2010 07:00:00,MOH0273,PC-6699,LOGON,Logon,logon


In [33]:
device_events = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    'DEVICE' AS event_type,
    activity AS event_details,
    'device' AS source
FROM read_csv_auto('{(BASE_PATH/'device.csv').as_posix()}')
LIMIT 100
""").df()

device_events.head()

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 07:21:06,MOH0273,PC-6699,DEVICE,Connect,device
1,01/02/2010 07:37:41,MOH0273,PC-6699,DEVICE,Disconnect,device
2,01/02/2010 07:59:11,HPH0075,PC-2417,DEVICE,Connect,device
3,01/02/2010 07:59:49,IIW0249,PC-0843,DEVICE,Connect,device
4,01/02/2010 08:04:26,IIW0249,PC-0843,DEVICE,Disconnect,device


In [34]:
email_events = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    'EMAIL' AS event_type,
    content AS event_details,
    'email' AS source
FROM read_csv_auto('{(BASE_PATH/'email.csv').as_posix()}')
LIMIT 100
""").df()

email_events.head()

,timestamp,user,pc,event_type,event_details,source
0,2010-02-01 07:11:45,LAP0338,PC-5758,EMAIL,middle f2 systems 4 july techniques powerful d...,email
1,2010-02-01 07:12:16,MOH0273,PC-6699,EMAIL,the breaking called allied reservations former...,email
2,2010-02-01 07:13:00,LAP0338,PC-5758,EMAIL,slowly this uncinus winter beneath addition ex...,email
3,2010-02-01 07:13:17,LAP0338,PC-5758,EMAIL,400 other difficult land cirrocumulus powered ...,email
4,2010-02-01 07:13:28,MOH0273,PC-6699,EMAIL,this kmh october holliswood number advised unu...,email


In [35]:
file_events = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    'FILE' AS event_type,
    filename AS event_details,
    'file' AS source
FROM read_csv_auto('{(BASE_PATH/'file.csv').as_posix()}')
LIMIT 100
""").df()

file_events.head()

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 07:23:14,MOH0273,PC-6699,FILE,EYPC9Y08.doc,file
1,01/02/2010 07:26:19,MOH0273,PC-6699,FILE,N3LTSU3O.pdf,file
2,01/02/2010 08:12:03,HPH0075,PC-2417,FILE,D3D3WC9W.doc,file
3,01/02/2010 08:17:00,HPH0075,PC-2417,FILE,QCSW62YS.doc,file
4,01/02/2010 08:24:57,HSB0196,PC-8001,FILE,AU75JV6U.jpg,file


In [36]:
http_events = duckdb.sql(f"""
SELECT
    date AS timestamp,
    user,
    pc,
    'HTTP' AS event_type,
    url AS event_details,
    'http' AS source
FROM read_csv_auto('{(BASE_PATH/'http.csv').as_posix()}')
LIMIT 100
""").df()

http_events.head()

,timestamp,user,pc,event_type,event_details,source
0,2010-02-01 06:55:16,LRR0148,PC-4275,HTTP,http://msn.com/The_Human_Centipede_First_Seque...,http
1,2010-02-01 07:00:13,NGF0157,PC-6056,HTTP,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...,http
2,2010-02-01 07:03:46,NGF0157,PC-6056,HTTP,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...,http
3,2010-02-01 07:05:26,IRM0931,PC-7188,HTTP,http://groupon.com/Leonhard_Euler/leonhard/tne...,http
4,2010-02-01 07:05:52,IRM0931,PC-7188,HTTP,http://flickr.com/Inauguration_of_Barack_Obama...,http


In [37]:
timeline = pd.concat(
    [
        logon_events,
        device_events,
        email_events,
        file_events,
        http_events
    ],
    ignore_index=True
)

timeline.head(20)

,timestamp,user,pc,event_type,event_details,source
0,01/02/2010 06:49:00,NGF0157,PC-6056,LOGON,Logon,logon
1,01/02/2010 06:50:00,LRR0148,PC-4275,LOGON,Logon,logon
2,01/02/2010 06:53:04,LRR0148,PC-4124,LOGON,Logon,logon
3,01/02/2010 07:00:00,IRM0931,PC-7188,LOGON,Logon,logon
4,01/02/2010 07:00:00,MOH0273,PC-6699,LOGON,Logon,logon
5,01/02/2010 07:07:00,LAP0338,PC-5758,LOGON,Logon,logon
6,01/02/2010 07:08:00,MHH0180,PC-9822,LOGON,Logon,logon
7,01/02/2010 07:08:00,NOB0181,PC-3446,LOGON,Logon,logon
8,01/02/2010 07:13:00,AHC0142,PC-8893,LOGON,Logon,logon
9,01/02/2010 07:14:00,CTR0341,PC-6184,LOGON,Logon,logon


In [38]:
timeline["timestamp"] = pd.to_datetime(
    timeline["timestamp"]
)

timeline = timeline.sort_values("timestamp")

timeline.head(20)

,timestamp,user,pc,event_type,event_details,source
0,2010-01-02 06:49:00,NGF0157,PC-6056,LOGON,Logon,logon
1,2010-01-02 06:50:00,LRR0148,PC-4275,LOGON,Logon,logon
2,2010-01-02 06:53:04,LRR0148,PC-4124,LOGON,Logon,logon
3,2010-01-02 07:00:00,IRM0931,PC-7188,LOGON,Logon,logon
4,2010-01-02 07:00:00,MOH0273,PC-6699,LOGON,Logon,logon
5,2010-01-02 07:07:00,LAP0338,PC-5758,LOGON,Logon,logon
6,2010-01-02 07:08:00,MHH0180,PC-9822,LOGON,Logon,logon
7,2010-01-02 07:08:00,NOB0181,PC-3446,LOGON,Logon,logon
8,2010-01-02 07:13:00,AHC0142,PC-8893,LOGON,Logon,logon
9,2010-01-02 07:14:00,CTR0341,PC-6184,LOGON,Logon,logon


In [39]:
TIMELINE_DIR = PROJECT_ROOT / "datasets" / "integrated"
TIMELINE_DIR.mkdir(parents=True, exist_ok=True)

timeline.to_parquet(
    TIMELINE_DIR / "employee_event_timeline_sample.parquet",
    index=False
)

print("Sample Timeline Saved Successfully!")

Sample Timeline Saved Successfully!
